In [ ]:
"""
Sistema de recomendação de cursos baseado no perfil profissional.

Entrada:
  - Área
  - Prioridade
  - Estilo de estudo

Saída:
  - Catálogo personalizado
  - Plano ótimo usando mochila
  - Relatório de horas, orçamento e relevância

Objetivo:
  Maximizar relevância dentro das restrições de horas e orçamento.

Documentação completa e explicação das funções no arquivo README.docx.
"""




from functools import lru_cache
import pandas as pd
from IPython.display import display, HTML

# ============================================================
# 1) Funções auxiliares para exibir no Jupyter
# ============================================================

def mostrar(texto):
    """Mostra texto formatado imediatamente no console do Jupyter."""
    display(HTML(f"<pre>{texto}</pre>"))

# ============================================================
# 2) CATÁLOGO DINÂMICO
# ============================================================

def montar_lista_cursos(area_foco):
    area_foco = area_foco.strip().lower()

    cursos_gerais = [
        {"curso": "Fundamentos de IA para Profissionais", "area": "Geral", "horas": 6, "preco": 500, "impacto": 8},
        {"curso": "Produtividade com IA no Dia a Dia", "area": "Geral", "horas": 4, "preco": 350, "impacto": 7},
        {"curso": "Prompt Engineering Aplicado", "area": "Geral", "horas": 5, "preco": 450, "impacto": 9},
    ]

    cursos_por_area = {
        "saúde": [
            {"curso": "IA para Análise de Exames", "area": "Saúde", "horas": 8, "preco": 700, "impacto": 9},
            {"curso": "Suporte a Diagnóstico com IA", "area": "Saúde", "horas": 10, "preco": 900, "impacto": 10},
        ],
        "direito": [
            {"curso": "Automação de Documentos Jurídicos com IA", "area": "Direito", "horas": 6, "preco": 650, "impacto": 8},
            {"curso": "Análise de Jurisprudência com IA", "area": "Direito", "horas": 9, "preco": 850, "impacto": 9},
        ],
        "educação": [
            {"curso": "Planejamento de Aulas com IA", "area": "Educação", "horas": 5, "preco": 500, "impacto": 8},
            {"curso": "Personalização de Atividades com IA", "area": "Educação", "horas": 7, "preco": 650, "impacto": 9},
        ],
        "engenharia": [
            {"curso": "Modelos de IA para Engenharia", "area": "Engenharia", "horas": 10, "preco": 950, "impacto": 10},
            {"curso": "Visão Computacional para Engenharia", "area": "Engenharia", "horas": 8, "preco": 800, "impacto": 9},
        ],
        "administração": [
            {"curso": "Dashboards de Negócios com IA", "area": "Administração", "horas": 7, "preco": 700, "impacto": 9},
            {"curso": "Análise de Dados de Vendas com IA", "area": "Administração", "horas": 6, "preco": 650, "impacto": 8},
        ],
    }

    catalogo = cursos_gerais.copy()

    if area_foco in cursos_por_area:
        catalogo.extend(cursos_por_area[area_foco])
    else:
        for lista in cursos_por_area.values():
            catalogo.append(lista[0])

    return catalogo


# ============================================================
# 3) PERFIL DO USUÁRIO (menus via display())
# ============================================================

def coletar_perfil_usuario():

    mostrar(
        "===== PERFIL DO PROFISSIONAL =====\n"
        "1 - Saúde\n"
        "2 - Direito\n"
        "3 - Educação\n"
        "4 - Engenharia\n"
        "5 - Administração\n"
        "6 - Todas / Genérico\n"
    )
    escolha = input("Digite sua área (1-6): ")

    mapa = {
        "1": "saúde",
        "2": "direito",
        "3": "educação",
        "4": "engenharia",
        "5": "administração",
        "6": "todas"
    }
    area_foco = mapa.get(escolha, "todas")

    mostrar("De 1 a 10, qual a prioridade de IA pra você?")
    prioridade = input("Digite (1-10): ")

    try:
        prioridade = int(prioridade)
    except:
        prioridade = 5
    prioridade = max(1, min(10, prioridade))

    mostrar(
        "===== ESTILO DE ESTUDO =====\n"
        "1 - Curto e barato\n"
        "2 - Equilibrado\n"
        "3 - Aprofundado\n"
    )
    estilo = input("Digite (1-3): ")

    if estilo == "1":
        return area_foco, prioridade, 8, 800
    elif estilo == "3":
        return area_foco, prioridade, 25, 3000
    return area_foco, prioridade, 15, 1500


# ============================================================
# 4) RELEVÂNCIA (simples e clara)
# ============================================================

def calcular_relevancia_cursos(cursos, area_foco, prioridade):
    cursos_novos = []

    for curso in cursos:
        base = curso["impacto"]

        bonus_area = 2 if curso["area"].lower() == area_foco else 1
        extra = prioridade // 5  # 0, 1 ou 2

        relevancia = min(10, base + bonus_area + extra)

        novo = curso.copy()
        novo["relevancia"] = relevancia
        cursos_novos.append(novo)

    return cursos_novos


# ============================================================
# 5) MERGE SORT
# ============================================================

def merge_sort_lista(lista, chave, memo):
    tupla = tuple(id(x) for x in lista)

    if tupla in memo:
        return memo[tupla]

    if len(lista) <= 1:
        memo[tupla] = lista
        return lista

    meio = len(lista)//2
    e = merge_sort_lista(lista[:meio], chave, memo)
    d = merge_sort_lista(lista[meio:], chave, memo)

    res = []
    i = j = 0
    while i < len(e) and j < len(d):
        if chave(e[i]) <= chave(d[j]):
            res.append(e[i]); i+=1
        else:
            res.append(d[j]); j+=1

    res.extend(e[i:])
    res.extend(d[j:])
    memo[tupla] = res
    return res


# ============================================================
# 6) MOCHILA (DP + recursão + memo)
# ============================================================

def montar_plano_otimo(horas_max, orcamento_max, area_foco, prioridade):
    cursos = montar_lista_cursos(area_foco)
    cursos = calcular_relevancia_cursos(cursos, area_foco, prioridade)

    memo_sort = {}
    cursos_ordenados = merge_sort_lista(
        cursos,
        lambda c: c["relevancia"] / c["horas"],
        memo_sort
    )

    @lru_cache(maxsize=None)
    def knapsack(i, h, o):
        if i == len(cursos_ordenados) or h == 0 or o == 0:
            return 0

        curso = cursos_ordenados[i]
        melhor = knapsack(i+1, h, o)

        if curso["horas"] <= h and curso["preco"] <= o:
            melhor = max(melhor,
                curso["relevancia"] + knapsack(i+1, h-curso["horas"], o-curso["preco"])
            )
        return melhor

    # Reconstruir solução
    escolhidos = []
    h = horas_max
    o = orcamento_max
    for i in range(len(cursos_ordenados)):
        if knapsack(i, h, o) != knapsack(i+1, h, o):
            escolhidos.append(cursos_ordenados[i])
            h -= cursos_ordenados[i]["horas"]
            o -= cursos_ordenados[i]["preco"]

    return (
        pd.DataFrame(cursos_ordenados),
        pd.DataFrame(escolhidos),
        knapsack(0, horas_max, orcamento_max),
        horas_max - h,
        orcamento_max - o
    )


# ============================================================
# 7) MENU PRINCIPAL PARA JUPYTER (corrigido!)
# ============================================================

def executar_sistema():

    while True:
        mostrar(
            "====================================\n"
            "   PLATAFORMA DE CURSOS DE IA\n"
            "====================================\n"
            "1 - Montar plano de estudos\n"
            "0 - Sair\n"
        )

        opcao = input("Escolha uma opção (0/1): ").strip()

        if opcao == "0":
            mostrar("Saindo...")
            break

        elif opcao == "1":
            area, prioridade, horas, orc = coletar_perfil_usuario()

            df_cat, df_sel, valor_max, h_usadas, gasto = montar_plano_otimo(
                horas, orc, area, prioridade
            )

            mostrar("===== CATÁLOGO DE CURSOS PERSONALIZADO =====")
            display(df_cat[["curso","area","horas","preco","relevancia"]])

            mostrar("===== MELHOR TRILHA PRA VOCÊ =====")
            if df_sel.empty:
                mostrar("Nenhum curso selecionado.")
            else:
                display(df_sel[["curso","area","horas","preco","relevancia"]])

            mostrar(
                f"===== RESUMO =====\n"
                f"Horas usadas: {h_usadas}\n"
                f"Orçamento gasto: R$ {gasto}\n"
                f"Relevância total: {valor_max}\n"
            )

        else:
            mostrar("Opção inválida! Tente novamente.")


# ============================================================
# 8) INICIAR
# ============================================================

executar_sistema()


,curso,area,horas,preco,relevancia
0,Suporte a Diagnóstico com IA,Saúde,10,900,10
1,IA para Análise de Exames,Saúde,8,700,10
2,Fundamentos de IA para Profissionais,Geral,6,500,10
3,Prompt Engineering Aplicado,Geral,5,450,10
4,Produtividade com IA no Dia a Dia,Geral,4,350,9


,curso,area,horas,preco,relevancia
0,Fundamentos de IA para Profissionais,Geral,6,500,10
1,Prompt Engineering Aplicado,Geral,5,450,10
2,Produtividade com IA no Dia a Dia,Geral,4,350,9
